<a href="https://colab.research.google.com/github/AnkaRezaa/machine_learning_254107023002/blob/main/jobsheet%20dataset%20AntiBurnOut/JS05/05_Clustering_HDBSCAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JS05: Advanced Hierarchical Clustering (HDBSCAN) - Dataset AntiBurnOut

**Identitas Mahasiswa:**
- **Nama**  : MUHAMMAD ANKA REZA
- **NIM**   : 254107023002
- **Kelas** : TI/3H

---

### Deskripsi Praktikum:
Pada praktikum ini, kita menerapkan algoritma clustering kepadatan hierarki tingkat lanjut (**HDBSCAN**) untuk mengelompokkan data kesehatan mental. Berbeda dengan K-Means yang mengasumsikan klaster berbentuk bola sferis, HDBSCAN mampu menemukan klaster dengan kepadatan dan bentuk arbitrer serta memisahkan outlier secara otomatis.

### Langkah 1: Memuat Data dan Library HDBSCAN

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Coba import hdbscan, fallback ke AgglomerativeClustering jika modul belum terinstall
try:
    import hdbscan
    has_hdbscan = True
except ImportError:
    from sklearn.cluster import AgglomerativeClustering
    has_hdbscan = False

dataset_candidates = [
    'mental_health_burnout_prediction_dataset.csv',
    '../mental_health_burnout_prediction_dataset.csv',
    'jobsheet dataset AntiBurnOut/mental_health_burnout_prediction_dataset.csv'
]
dataset_path = next((p for p in dataset_candidates if os.path.exists(p)), 'mental_health_burnout_prediction_dataset.csv')
df = pd.read_csv(dataset_path)

features = ['Work_Hours_Per_Week', 'Sleep_Hours', 'Anxiety_Score', 'Depression_Score', 'Mood_Score']
X = df[features].copy()
for col in features:
    X[col] = X[col].fillna(X[col].median())

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Ambil 3.000 sampel data
np.random.seed(42)
sample_idx = np.random.choice(len(X_scaled), 3000, replace=False)
X_sample = X_scaled[sample_idx]

print("Data sampel siap diproses. Shape:", X_sample.shape)

### Langkah 2: Melatih Model HDBSCAN / Hierarchical Clustering

In [ ]:
if has_hdbscan:
    clusterer = hdbscan.HDBSCAN(min_cluster_size=50, min_samples=15, gen_min_span_tree=True)
    labels = clusterer.fit_predict(X_sample)
    method_name = 'HDBSCAN'
else:
    clusterer = AgglomerativeClustering(n_clusters=3)
    labels = clusterer.fit_predict(X_sample)
    method_name = 'Agglomerative Hierarchical'

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
print(f"{method_name} mendeteksi {n_clusters} klaster padat.")
if -1 in labels:
    print(f"Jumlah titik outlier (noise): {list(labels).count(-1)}")

### Langkah 3: Visualisasi Hasil Klaster dalam Ruang Reduksi Dimensi (PCA)

In [ ]:
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_sample)

plt.figure(figsize=(9, 6))
sns.scatterplot(x=X_2d[:, 0], y=X_2d[:, 1], hue=labels, palette='tab10', alpha=0.7)
plt.title(f'Visualisasi Klaster {method_name} Menggunakan PCA', fontsize=12)
plt.xlabel('PC 1')
plt.ylabel('PC 2')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

### Kesimpulan JS05:
1. Clustering berbasis hierarki memberikan pemisahan yang lebih fleksibel terhadap struktur densitas responden.
2. Responden dengan perpaduan nilai ekstrem (misal jam kerja sangat tinggi dengan waktu tidur sangat minim) dipisahkan dari kelompok populasi mayoritas.